# 💧 Étiage / Stress hydrique — LowFlow (CLIMADA)

Exploration du module `LowFlow` de CLIMADA pour modéliser le risque de pénurie d'eau à partir de données de débit hydrologique.

**Aléa** : Étiage / stress hydrique (LF)  
**Zone** : Bassins versants français  
**Données** : ISIMIP hydrological discharge / synthétique  
**Métriques** : Débit en dessous du seuil, durée, sévérité

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from climada.hazard import Hazard, Centroids
from climada.entity import Exposures, ImpactFuncSet, ImpactFunc
from climada.engine import Impact

try:
    from climada.hazard import LowFlow
    PETALS_OK = True
except ImportError:
    PETALS_OK = False
    print("⚠️ climada_petals non disponible — on utilisera des données synthétiques")

print(f"CLIMADA petals disponible : {PETALS_OK}")

## 1. Aléa — Étiage (Low Flow)

Le module `LowFlow` utilise les données de **débit hydrologique** ISIMIP pour identifier les périodes où le débit passe en dessous d'un **seuil critique** (percentile bas).

### Concept
- **Seuil** : Q10 ou Q20 (10ᵉ ou 20ᵉ percentile du débit historique)
- **Intensité** : déficit de débit en dessous du seuil (m³/s ou relatif)
- **Durée** : nombre de jours consécutifs sous le seuil
- **Sévérité** : déficit × durée (volume manquant)

### Données ISIMIP
Modèles hydrologiques globaux (H08, WaterGAP, etc.) forcés par les GCMs CMIP6.

In [ ]:
# --- CLIMADA LowFlow (nécessite données ISIMIP) ---
# from climada.hazard import LowFlow
# low_flow = LowFlow.from_netcdf(
#     input_dir='/path/to/isimip/',
#     countries=['FRA'],
#     bbox=(-5, 42, 10, 51),
#     percentile=10,
#     yearrange=(1980, 2020),
#     gh_model='h08',
#     cl_model='gfdl-esm4',
#     scenario='historical'
# )

# --- Données synthétiques : déficit de débit annuel ---
from scipy.sparse import csr_matrix, vstack

np.random.seed(42)

# Grille France : 42-51°N, -5 à 10°E
lon = np.arange(-5, 10.5, 0.5)
lat = np.arange(42, 51.5, 0.5)
lon_grid, lat_grid = np.meshgrid(lon, lat)
lon_flat = lon_grid.flatten()
lat_flat = lat_grid.flatten()
n_centroids = len(lon_flat)

centroids = Centroids.from_lat_lon(lat_flat, lon_flat)

# Principaux bassins versants (zones de débit)
basins = {
    'Seine': {'lon': 2.5, 'lat': 48.5, 'radius': 1.5, 'base_flow': 300},
    'Loire': {'lon': 1.0, 'lat': 47.0, 'radius': 2.0, 'base_flow': 350},
    'Rhône': {'lon': 5.0, 'lat': 45.0, 'radius': 1.5, 'base_flow': 500},
    'Garonne': {'lon': 0.5, 'lat': 44.0, 'radius': 1.5, 'base_flow': 200},
    'Rhin': {'lon': 7.5, 'lat': 48.5, 'radius': 1.0, 'base_flow': 400},
}

# Calculer le débit de référence par cellule
base_flow = np.ones(n_centroids) * 50  # débit minimal partout
for basin, params in basins.items():
    dist = np.sqrt((lon_flat - params['lon'])**2 + (lat_flat - params['lat'])**2)
    contribution = params['base_flow'] * np.exp(-0.5 * (dist / params['radius'])**2)
    base_flow += contribution

# Seuil Q10 : 30% du débit moyen (simplification)
q10_threshold = base_flow * 0.3

# Générer 40 ans d'étiage
n_years = 40
years = np.arange(1981, 1981 + n_years)
frequency = np.ones(n_years) / n_years

# Tendance : baisse des débits estivaux due au réchauffement
trend = np.linspace(0, -0.15, n_years)

intensity_list = []
for y in range(n_years):
    # Débit estival simulé (fraction du débit moyen)
    summer_fraction = 0.4 + np.random.normal(0, 0.1, n_centroids) + trend[y]
    
    # Gradient sud = plus d'étiage
    lat_factor = (lat_flat - 51) / (42 - 51)
    summer_fraction -= lat_factor * 0.1
    
    # Années de sécheresse exceptionnelle
    if y in [22, 35, 38, 39]:  # 2003, 2016, 2019, 2020
        summer_fraction -= np.random.uniform(0.1, 0.2, n_centroids)
    
    summer_flow = base_flow * np.clip(summer_fraction, 0.05, 1.0)
    
    # Déficit = max(0, seuil - débit) / seuil  (relatif)
    deficit = np.maximum(q10_threshold - summer_flow, 0) / q10_threshold
    deficit[deficit < 0.05] = 0  # seuil de détection
    
    intensity_list.append(csr_matrix(deficit))

intensity_matrix = vstack(intensity_list)

lowflow_haz = Hazard(
    haz_type='LF',
    centroids=centroids,
    intensity=intensity_matrix,
    frequency=frequency,
    event_id=np.arange(1, n_years + 1),
    event_name=[f'LowFlow_{y}' for y in years],
    date=pd.date_range('1981-08-01', periods=n_years, freq='365D').to_numpy().astype('datetime64[ns]').astype(int) // 10**9,
    units='relative deficit'
)

print(f"Hazard : {lowflow_haz.size} années")
print(f"Déficit max relatif : {lowflow_haz.intensity.max():.3f}")
print(f"Cellules en étiage (total) : {lowflow_haz.intensity.nnz}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Débit de référence
sc0 = axes[0].scatter(lon_flat, lat_flat, c=base_flow, s=15, cmap='Blues', vmin=0)
axes[0].set_title('Débit moyen de référence (m³/s)')
plt.colorbar(sc0, ax=axes[0], label='m³/s', shrink=0.8)
axes[0].set_ylabel('Latitude')

# Année 2003 (sécheresse)
idx_2003 = 22
deficit_2003 = lowflow_haz.intensity[idx_2003].toarray().flatten()
sc1 = axes[1].scatter(lon_flat, lat_flat, c=deficit_2003, s=15, cmap='YlOrRd', vmin=0, vmax=1)
axes[1].set_title(f'Déficit relatif — {years[idx_2003]}')
plt.colorbar(sc1, ax=axes[1], label='Déficit relatif', shrink=0.8)

# Fréquence d'étiage
freq_lowflow = np.array((lowflow_haz.intensity > 0.3).sum(axis=0)).flatten() / n_years * 100
sc2 = axes[2].scatter(lon_flat, lat_flat, c=freq_lowflow, s=15, cmap='RdYlGn_r', vmin=0)
axes[2].set_title('Fréquence étiage sévère (>30% déficit)')
plt.colorbar(sc2, ax=axes[2], label='% années', shrink=0.8)

for ax in axes:
    ax.set_xlabel('Longitude')
plt.tight_layout()
plt.show()

## 2. Analyse par bassin versant

Évolution temporelle du déficit hydrique par grand bassin versant français.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, (basin_name, params) in enumerate(basins.items()):
    ax = axes[idx // 3, idx % 3]
    
    # Cellules du bassin (rayon ~1.5°)
    dist = np.sqrt((lon_flat - params['lon'])**2 + (lat_flat - params['lat'])**2)
    basin_mask = dist < params['radius']
    
    if basin_mask.sum() == 0:
        continue
    
    # Déficit moyen par année
    mean_deficit = np.array(lowflow_haz.intensity[:, basin_mask].mean(axis=1)).flatten()
    
    colors = ['green' if v < 0.1 else 'orange' if v < 0.3 else 'red' for v in mean_deficit]
    ax.bar(years, mean_deficit, color=colors, alpha=0.8)
    
    z = np.polyfit(years, mean_deficit, 1)
    ax.plot(years, np.polyval(z, years), 'k--', linewidth=2,
            label=f'Tendance : {z[0]:+.5f}/an')
    
    ax.set_title(f'Bassin {basin_name}')
    ax.set_xlabel('Année')
    ax.set_ylabel('Déficit relatif moyen')
    ax.legend(fontsize=8)
    ax.set_ylim(0, 0.8)

# Dernier subplot : comparaison des bassins
ax = axes[1, 2]
for basin_name, params in basins.items():
    dist = np.sqrt((lon_flat - params['lon'])**2 + (lat_flat - params['lat'])**2)
    basin_mask = dist < params['radius']
    if basin_mask.sum() == 0:
        continue
    mean_deficit = np.array(lowflow_haz.intensity[:, basin_mask].mean(axis=1)).flatten()
    rolling = pd.Series(mean_deficit).rolling(5, min_periods=1).mean()
    ax.plot(years, rolling, linewidth=2, label=basin_name)

ax.set_title('Comparaison bassins (moy. glissante 5 ans)')
ax.set_xlabel('Année')
ax.set_ylabel('Déficit relatif')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Exposition — Usages de l'eau

L'exposition représente les usages dépendants du débit fluvial : irrigation agricole, industrie, eau potable, centrales thermiques.

In [ ]:
from shapely.geometry import Point
import geopandas as gpd

n_assets = 250
np.random.seed(321)

# Types d'usage de l'eau
water_users = {
    'irrigation': {'weight': 0.40, 'zones': [(1.0, 47.5), (0.0, 44.5), (4.5, 43.5), (3.0, 48.0)], 'value_range': (500_000, 3_000_000)},
    'industrie': {'weight': 0.25, 'zones': [(2.35, 48.86), (4.83, 45.76), (7.5, 48.5), (1.44, 43.6)], 'value_range': (5_000_000, 30_000_000)},
    'eau_potable': {'weight': 0.20, 'zones': [(2.35, 48.86), (5.37, 43.3), (-0.58, 44.84), (-1.55, 47.22)], 'value_range': (2_000_000, 15_000_000)},
    'centrales': {'weight': 0.15, 'zones': [(4.8, 44.9), (2.5, 47.3), (0.3, 47.5), (7.0, 48.0)], 'value_range': (20_000_000, 100_000_000)},
}

lons, lats, values, usage_types = [], [], [], []

for usage, params in water_users.items():
    n = int(n_assets * params['weight'])
    for zone_lon, zone_lat in params['zones']:
        n_zone = n // len(params['zones'])
        lons.extend(np.random.normal(zone_lon, 0.5, n_zone))
        lats.extend(np.random.normal(zone_lat, 0.3, n_zone))
        values.extend(np.random.uniform(*params['value_range'], n_zone))
        usage_types.extend([usage] * n_zone)

remaining = n_assets - len(lons)
if remaining > 0:
    lons.extend(np.random.uniform(-3, 8, remaining))
    lats.extend(np.random.uniform(43, 50, remaining))
    values.extend(np.random.uniform(1_000_000, 5_000_000, remaining))
    usage_types.extend(['irrigation'] * remaining)

gdf = gpd.GeoDataFrame({
    'value': np.array(values[:n_assets]),
    'impf_LF': np.ones(n_assets, dtype=int),
    'usage': usage_types[:n_assets],
    'geometry': [Point(lo, la) for lo, la in zip(lons[:n_assets], lats[:n_assets])]
}, crs='EPSG:4326')

exposure = Exposures(gdf)
exposure.check()

for usage in water_users:
    mask = exposure.gdf['usage'] == usage
    print(f"{usage:12s} : {mask.sum():3d} actifs, valeur = {exposure.gdf.loc[mask, 'value'].sum():>14,.0f} USD")
print(f"{'Total':12s} : {len(exposure.gdf):3d} actifs, valeur = {exposure.gdf['value'].sum():>14,.0f} USD")

## 4. Vulnérabilité — Fonctions d'impact étiage

L'impact de l'étiage dépend du type d'usage :
- **Irrigation** : impact direct et proportionnel au déficit
- **Industrie** : seuil de fonctionnement → step function
- **Centrales thermiques** : restrictions de refroidissement au-delà d'un seuil

In [ ]:
intensity_range = np.arange(0, 1.01, 0.01)

# Irrigation : linéaire
mdr_irrigation = np.clip(intensity_range * 1.2, 0, 1.0)
mdr_irrigation[intensity_range < 0.05] = 0

# Industrie : step function (seuil + plateau)
mdr_industry = np.where(intensity_range > 0.3, 
                         np.clip(0.3 + (intensity_range - 0.3) * 1.5, 0, 0.8), 0)

# Centrales : seuil strict (arrêt au-delà de 50% de déficit)
mdr_power = np.where(intensity_range > 0.5,
                      np.clip((intensity_range - 0.5) * 3.0, 0, 1.0), 0)

impf_irrigation = ImpactFunc(
    id=1, haz_type='LF',
    intensity=intensity_range, mdd=mdr_irrigation, paa=np.ones_like(mdr_irrigation),
    intensity_unit='relative deficit', name='Irrigation'
)

impf_industry = ImpactFunc(
    id=2, haz_type='LF',
    intensity=intensity_range, mdd=mdr_industry, paa=np.ones_like(mdr_industry),
    intensity_unit='relative deficit', name='Industrie'
)

impf_power = ImpactFunc(
    id=3, haz_type='LF',
    intensity=intensity_range, mdd=mdr_power, paa=np.ones_like(mdr_power),
    intensity_unit='relative deficit', name='Centrales'
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(intensity_range, mdr_irrigation, 'g-', linewidth=2, label='Irrigation (linéaire)')
ax.plot(intensity_range, mdr_industry, 'b--', linewidth=2, label='Industrie (seuil 30%)')
ax.plot(intensity_range, mdr_power, 'r:', linewidth=2, label='Centrales (seuil 50%)')
ax.set_xlabel('Déficit relatif de débit')
ax.set_ylabel('MDR')
ax.set_title('Fonctions de vulnérabilité — Étiage par usage')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Impact — Calcul des pertes par usage

In [ ]:
# On utilise la même impf_id=1 pour tous (simplifié)
impf_set = ImpactFuncSet([impf_irrigation])

imp = Impact()
imp.calc(exposure, impf_set, lowflow_haz, save_mat=True)

print(f"EAI total : {imp.aai_agg:,.0f} USD")
print(f"Perte max : {imp.at_event.max():,.0f} USD")

# Pertes temporelles
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(years, imp.at_event / 1e6, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Année')
axes[0].set_ylabel('Perte (M USD)')
axes[0].set_title('Pertes annuelles — Étiage')
axes[0].grid(True, alpha=0.3, axis='y')

fc = imp.calc_freq_curve()
axes[1].plot(fc.return_per, fc.impact / 1e6, 'b-', linewidth=2)
axes[1].set_xlabel('Période de retour (ans)')
axes[1].set_ylabel('Perte (M USD)')
axes[1].set_title('Courbe de fréquence')
axes[1].set_xscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

eai_exp = imp.eai_exp
gdf_plot = exposure.gdf.copy()
gdf_plot['eai'] = eai_exp

usage_markers = {'irrigation': 's', 'industrie': 'o', 'eau_potable': '^', 'centrales': 'D'}
usage_colors = {'irrigation': 'green', 'industrie': 'blue', 'eau_potable': 'cyan', 'centrales': 'red'}

for usage, marker in usage_markers.items():
    mask = gdf_plot['usage'] == usage
    if mask.sum() == 0:
        continue
    subset = gdf_plot[mask]
    ax.scatter(subset.geometry.x, subset.geometry.y, c=subset['eai'],
              marker=marker, s=30, cmap='YlOrRd', vmin=0, vmax=gdf_plot['eai'].quantile(0.95),
              label=usage, alpha=0.7)

ax.set_title('EAI par actif et usage — Étiage')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 6. Synthèse

| Aspect | Détail |
|--------|--------|
| Source données | ISIMIP hydrological discharge (H08, WaterGAP) |
| Variable | Débit journalier (m³/s) → déficit sous Q10/Q20 |
| Résolution | 0.5° (~50 km) |
| Seuil | Percentile historique (Q10, Q20) |
| Particularité | Impact sectoriel très différencié |

### Points clés
- Les bassins Garonne et Loire sont les plus vulnérables à l'étiage estival
- Les centrales thermiques/nucléaires ont un seuil critique : au-delà de 50% de déficit, arrêt obligatoire
- Tendance climatique : baisse des débits estivaux de 10-30% d'ici 2050 (CMIP6)
- L'irrigation représente le plus grand volume d'usage mais l'industrie le plus grand impact économique

### Prochaines étapes
- Intégrer les données ISIMIP réelles (multi-modèle hydrologique)
- Coupler avec les projections CMIP6 (scénarios SSP)
- Analyser la saisonnalité (étiage estival vs hivernal)
- Explorer les conflits d'usage (irrigation vs énergie vs eau potable)